# 18 Final Conclusion And Best Model

This is the final reporting notebook for the active hourly DA pipeline.

It does three things:
- gather the latest comparable saved `FS2` and all-exogenous `FS3` challenger runs
- select the final challenger on **validation only**
- display the held-out test results and objective case-week plots for the selected winner

The grouped-ablation interpretation now lives in notebook `17_feature_family_value_results.ipynb`, with model-specific `FS3` ablation diagnostics in notebooks `15` and `16`.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [ ]:
from hourly_da.notebook_support import (
    build_compared_models_overview,
    build_model_style_map,
    build_reporting_summary_table,
    build_week_metrics_for_predictions,
    filter_available_comparison_specs,
    load_standard_report_bundle,
    render_plot_gallery,
    style_model_overview_table,
    style_reporting_summary_table,
    write_standard_week_selection_plots,
)


In [ ]:
rows = []
for run_label in ['model_comparison', 'lear_fs2_benchmark', 'xgboost_fs2_benchmark', 'prophet_fs2_benchmark', 'fs3_combo_promoted_confirm', 'case_week_selection']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


## Combined comparison summary

The final challenger is chosen on the validation split using the full-horizon reporting level. The test split is shown only after that choice is frozen.


In [ ]:
comparison_run_dir = (
    latest_run_or_none("fs3_combo_promoted_confirm")
    or latest_run_or_none("model_comparison")
)
report_bundle = None
selected_model = None
selected_display = None
official_naive_model = None
model_styles = {}
model_name_order = []

if comparison_run_dir is None:
    print("No comparable FS2 or FS3 run exists yet.")
else:
    official_naive = load_json(comparison_run_dir, "official_naive_reference.json")
    official_naive_model = str(official_naive["model"])
    official_naive_display = f"Naive benchmark ({official_naive_model.replace('naive_', '').replace('_', ' ').title()})"

    comparison_specs = [
        {
            "model": official_naive_model,
            "display_name": official_naive_display,
            "description": "Validation-selected seasonal naive benchmark.",
            "role": "benchmark",
        },
    ]
    comparison_specs.extend(
        filter_available_comparison_specs(
            output_root=output_root,
            current_run_dir=comparison_run_dir,
            candidate_specs=[
                {
                    "model": "lear_fs2",
                    "run_label": "lear_fs2_benchmark",
                    "display_name": "LEAR FS2",
                    "description": "Shortlisted linear benchmark from notebook 08.",
                    "role": "challenger",
                },
                {
                    "model": "xgboost_fs2",
                    "run_label": "xgboost_fs2_benchmark",
                    "display_name": "XGBoost FS2",
                    "description": "Shortlisted tree benchmark from notebook 09.",
                    "role": "challenger",
                },
                {
                    "model": "prophet_fs2",
                    "run_label": "prophet_fs2_benchmark",
                    "display_name": "Prophet FS2",
                    "description": "Additive benchmark from notebook 10 when available.",
                    "role": "challenger",
                },
                {
                    "model": "lear_fs3_combo_promoted",
                    "source": "current",
                    "display_name": "LEAR FS3",
                    "description": "FS3 linear parent with the combined all-exogenous stack.",
                    "role": "challenger",
                },
                {
                    "model": "xgboost_fs3_combo_promoted",
                    "source": "current",
                    "display_name": "XGBoost FS3",
                    "description": "FS3 tree parent with the combined all-exogenous stack.",
                    "role": "challenger",
                },
            ],
        )
    )

    seen_models = set()
    deduped_specs = []
    for spec in comparison_specs:
        model_name = str(spec["model"])
        if model_name in seen_models:
            continue
        seen_models.add(model_name)
        deduped_specs.append(spec)
    comparison_specs = deduped_specs

    if not comparison_specs:
        print("No challenger models were available for the final conclusion notebook.")
    else:
        report_bundle = load_standard_report_bundle(
            output_root=output_root,
            current_run_dir=comparison_run_dir,
            comparison_specs=comparison_specs,
        )
        comparison_specs_frame = report_bundle["comparison_specs"].copy()
        model_styles = build_model_style_map(
            comparison_specs_frame,
            official_naive_model=str(report_bundle["official_naive"]["model"]),
        )

        active_families = (
            model_status_frame()
            .loc[lambda df: df["status"] == "Active", "model_family"]
            .astype(str)
            .tolist()
        )
        comparison_specs_frame = comparison_specs_frame[
            comparison_specs_frame["model_family"].isin(active_families)
        ].copy()
        comparison_specs_frame = comparison_specs_frame.sort_values(["comparison_order", "display_name"]).reset_index(drop=True)
        model_name_order = comparison_specs_frame["model"].astype(str).tolist()
        display_name_order = comparison_specs_frame["display_name"].astype(str).tolist()

        overview = build_compared_models_overview(
            comparison_specs_frame,
            official_naive_model=str(report_bundle["official_naive"]["model"]),
        )
        display(style_model_overview_table(overview))

        active_metrics = report_bundle["metrics_by_reporting_level"][
            report_bundle["metrics_by_reporting_level"]["model"].isin(model_name_order)
        ].copy()
        validation_summary = build_reporting_summary_table(
            metrics_by_reporting_level=active_metrics,
            split_name="validation",
            model_order=display_name_order,
        )
        test_summary = build_reporting_summary_table(
            metrics_by_reporting_level=active_metrics,
            split_name="test",
            model_order=display_name_order,
        )
        display(style_reporting_summary_table(validation_summary, caption="Validation summary"))
        display(style_reporting_summary_table(test_summary, caption="Test summary"))

        selection_frame = active_metrics[
            (active_metrics["dataset_split"] == "validation")
            & (active_metrics["reporting_level"] == "stitched_all_horizon")
            & (active_metrics["model_family"].isin(["lear", "xgboost", "prophet"]))
        ].copy()

        if selection_frame.empty:
            print("No active challenger models were found in the current comparison bundle.")
        else:
            selection_frame["abs_bias"] = selection_frame["bias"].astype(float).abs()
            selection_frame = selection_frame.sort_values(["mae", "rmse", "abs_bias", "comparison_order", "model"])
            validation_row = selection_frame.iloc[0]
            selected_model = str(validation_row["model"])
            selected_display = str(validation_row.get("display_name", selected_model))

            test_row = active_metrics[
                (active_metrics["dataset_split"] == "test")
                & (active_metrics["reporting_level"] == "stitched_all_horizon")
                & (active_metrics["model"].astype(str) == selected_model)
            ].copy()
            test_row = test_row.sort_values(["mae", "model"]).head(1)

            conclusion_row = {
                "selected_final_model": selected_display,
                "selected_internal_model": selected_model,
                "naive_benchmark": official_naive_model,
                "validation_mae": float(validation_row["mae"]),
                "validation_rmae": float(validation_row["rmae_vs_official_naive"]),
                "test_mae": float(test_row.iloc[0]["mae"]) if not test_row.empty else float("nan"),
                "test_rmae": float(test_row.iloc[0]["rmae_vs_official_naive"]) if not test_row.empty else float("nan"),
                "test_rmse": float(test_row.iloc[0]["rmse"]) if not test_row.empty else float("nan"),
                "test_bias": float(test_row.iloc[0]["bias"]) if not test_row.empty else float("nan"),
            }

            display(
                Markdown(
                    f"### Selected final challenger
"
                    f"`{selected_display}` is selected on **validation** full-horizon MAE. "
                    f"The test rows below are shown only after that selection is frozen."
                )
            )
            display(pd.DataFrame([conclusion_row]))

            dm_rows = report_bundle["diebold_mariano_by_reporting_level"].copy()
            dm_rows = dm_rows[
                (dm_rows["challenger_model"].astype(str) == selected_model)
                & (dm_rows["benchmark_model"].astype(str) == official_naive_model)
            ].copy()
            if "reporting_level" in dm_rows.columns:
                dm_rows = dm_rows[dm_rows["reporting_level"].astype(str) != "guidance_only"].copy()
            if dm_rows.empty:
                print("No Diebold-Mariano rows were found for the selected final challenger.")
            else:
                display(
                    dm_rows[
                        [
                            "dataset_split",
                            "reporting_level_label",
                            "n_obs",
                            "dm_stat",
                            "p_value",
                        ]
                    ]
                    .sort_values(["dataset_split", "reporting_level_label"])
                    .reset_index(drop=True)
                )


## Objective case weeks

Only the required thesis cases are displayed here:
- typical winter
- typical summer
- high volatility


In [ ]:
required_categories = ["typical_winter", "typical_summer", "high_volatility"]

if report_bundle is None:
    print("No comparison bundle is available yet for the final notebook.")
elif selected_model is None or official_naive_model is None:
    print("No final challenger has been selected yet.")
else:
    _, selected_weeks = load_selected_case_weeks(output_root)
    if "category" in selected_weeks.columns:
        selected_weeks = selected_weeks[selected_weeks["category"].isin(required_categories)].copy()

    display(selected_weeks)

    focus_models = [official_naive_model, selected_model]
    week_metrics = build_week_metrics_for_predictions(
        predictions=report_bundle["predictions_long"],
        config=config,
        selected_weeks=selected_weeks,
        split_name="test",
        reporting_level="d_only",
        models=focus_models,
    )
    if not week_metrics.empty:
        focus_model_order = {model_name: position for position, model_name in enumerate(focus_models)}
        display(
            week_metrics
            .assign(_model_order=lambda frame: frame["model"].map(focus_model_order).fillna(len(focus_model_order)))
            .sort_values(["category", "_model_order", "model"])
            .drop(columns=["_model_order"])
            .reset_index(drop=True)
        )

    final_output_dir = output_root / "notebook_artifacts" / "18_final_conclusion_and_best_model" / (comparison_run_dir.name if comparison_run_dir is not None else "no_run")
    final_output_dir.mkdir(parents=True, exist_ok=True)
    overlay_paths = write_standard_week_selection_plots(
        predictions=report_bundle["predictions_long"],
        config=config,
        selected_weeks=selected_weeks,
        output_dir=final_output_dir / "selected_model_weeks",
        model_order=focus_models,
        model_styles=model_styles,
        split_name="test",
        reporting_level="d_only",
        title_prefix="Final challenger vs naive benchmark",
    )
    if not overlay_paths:
        print("No case-week overlays could be generated for the selected model.")
    else:
        display(render_plot_gallery(overlay_paths, columns=1))
